# Sequential PPM — Rotated Patches, Explicit Routes

Joint Pauli-product measurements (`M(P̄₁⊗P̄₂⊗…)`) on rotated surface-code patches placed on the seam-column coarse grid (pitch `2d+2`).
Each `PPMStep` **requires an explicit `route`**: the corridor's coarse cells, in order — `[]` for cell-adjacent targets (there is no auto-router in this package).

Cell-adjacent pairs are classified live by the four-row rule table: rows 1/4 → zero-cell merge, rows 2/3 → stretched-stabilizer wall (diagonal schedule).

The `lightstim.protocols.ppm` package is copied from the author's [CircLS repository](https://github.com/John-YuehanZhang/CircLS) @ `8802a5b` (minimal explicit-route variant, independent of CircLS).

In [ ]:
import contextlib, io

from lightstim.noise.config import NoiseConfig
from lightstim.protocols import PatchSpec, PPMStep, origin_of, SequentialPPMExperiment

D = 3
NP = NoiseConfig(p_1q=1e-3, p_2q=1e-3, p_meas=1e-3, p_reset=1e-3, p_idle=1e-3)

def spec(nm, a, b, o):
    return PatchSpec(nm, origin_of(a, b, D, seam=True), D, o)

def build(px, seq, states, **kw):
    exp = SequentialPPMExperiment(px, seq, initial_states=states,
                                  final_measure_states=states,
                                  rounds=D, rounds_init=1, **kw)
    with contextlib.redirect_stdout(io.StringIO()):
        c = exp.build()
    det, obs = c.compile_detector_sampler(seed=0).sample(512, separate_observables=True)
    noisy = exp.builder.build_noisy_circuit(noise_params=NP, noise_model='circuit_level')
    gl = len(noisy.shortest_graphlike_error())
    print(f'observables={c.num_observables}  det_silent={not det.any()}  '
          f'obs_deterministic={not obs.any()}  graphlike_distance={gl}')
    return exp, c

## Example 1 — one-cell corridor `Z̄⊗Z̄`

Two patches one coarse cell apart; the corridor occupies cell `(1, 0)`. Merge `d` rounds, measure out the corridor only, then read both patches out.

In [ ]:
exp1, c1 = build([spec('A', 0, 0, 'X_horizontal'), spec('B', 2, 0, 'X_horizontal')],
                 [PPMStep([('A', 'Z'), ('B', 'Z')], route=[(1, 0)])],
                 {'A': 'Z', 'B': 'Z'})

In [ ]:
c1.diagram('detslice-with-ops-svg')

## Example 2 — cell-adjacent pair: the rule table decides

Same measured letter on both patches (`route=[]`):
* same weight-2 positions → **row 1**, plain merge (bent schedule);
* different positions (here via `colour_swapped`) → **row 2**, uniform-domino stretched wall (diagonal schedule, no corridor data qubits).

In [ ]:
exp2a, c2a = build([spec('A', 0, 0, 'X_horizontal'), spec('B', 1, 0, 'X_horizontal')],
                   [PPMStep([('A', 'Z'), ('B', 'Z')], route=[])],
                   {'A': 'Z', 'B': 'Z'})
print('rule:', exp2a._rules[0].reason, '| schedule:', exp2a._sched[0])

In [ ]:
exp2b, c2b = build([spec('A', 0, 0, 'X_horizontal'), spec('B', 0, 1, 'X_vertical')],
                   [PPMStep([('A', 'X'), ('B', 'X')], route=[])],
                   {'A': 'X', 'B': 'X'}, colour_swapped={'B'})
print('rule:', exp2b._rules[0].reason, '| schedule:', exp2b._sched[0])

In [ ]:
c2b.diagram('detslice-with-ops-svg')

## Example 3 — three targets in one step (T corridor)

One PPM measures `Z̄⊗Z̄⊗Z̄` on three patches through a 3-cell corridor; the joint measurement fixes two independent pairwise products (`observables=2`).

In [ ]:
exp3, c3 = build([spec('q1', 0, 0, 'X_horizontal'), spec('q2', 4, 0, 'X_horizontal'),
                  spec('q3', 2, 1, 'X_vertical')],
                 [PPMStep([('q1', 'Z'), ('q2', 'Z'), ('q3', 'Z')],
                          route=[(1, 0), (2, 0), (3, 0)])],
                 {'q1': 'Z', 'q2': 'Z', 'q3': 'Z'})

In [ ]:
c3.diagram('detslice-with-ops-svg')

## Example 4 — a two-step sequence

PPMs run back to back on the same patches: between steps only the corridor is measured out — the patches persist with state.

In [ ]:
exp4, c4 = build([spec('A', 0, 0, 'X_horizontal'), spec('B', 2, 0, 'X_horizontal'),
                  spec('C', 4, 0, 'X_horizontal')],
                 [PPMStep([('A', 'Z'), ('B', 'Z')], route=[(1, 0)]),
                  PPMStep([('B', 'Z'), ('C', 'Z')], route=[(3, 0)])],
                 {'A': 'Z', 'B': 'Z', 'C': 'Z'})

In [ ]:
c4.diagram('detslice-with-ops-svg')